# 0. Install and Import Dependencies

In [2]:
import cv2
import mediapipe as mp
import numpy as np
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
import os
import csv
import math
from collections import deque
print(cv2.__version__)
import pandas as pd
print(pd.__version__)
from mediapipe.python.solutions.pose import PoseLandmark

4.10.0
2.2.2


# 1. Make Detections

# 2. Determining Joints

<img src="https://i.imgur.com/3j8BPdc.png" style="height:300px" >

In [6]:
for lndmrk in mp_pose.PoseLandmark:
    print(lndmrk)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32


# 3. Calculate Angles

In [8]:
def calculate_angle(a,b,c):
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle >180.0:
        angle = 360-angle
        
    return angle 

def calculate_angle_3d(a, b, c):
    """
    Calculates the 3D angle ABC (in degrees).
    a, b, c must be iterable with length 3: [x, y, z]
    """
    a = np.array(a, dtype=np.float32)
    b = np.array(b, dtype=np.float32)
    c = np.array(c, dtype=np.float32)

    ba = a - b
    bc = c - b

    ba_norm = np.linalg.norm(ba)
    bc_norm = np.linalg.norm(bc)

    if ba_norm == 0 or bc_norm == 0:
        return np.nan

    cos_angle = np.dot(ba, bc) / (ba_norm * bc_norm)
    cos_angle = np.clip(cos_angle, -1.0, 1.0)

    angle = np.degrees(np.arccos(cos_angle))
    return angle



In [9]:
bowling_filename = <path to bowler video>

#code below is for video capture
cap = cv2.VideoCapture (bowling_filename + ".mp4")

#code below is for Camera Logitech
#cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
#cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)   # or 1920
#cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)   # or 1080
#cap.set(cv2.CAP_PROP_FPS, 30)

if not cap.isOpened():
    print("Error: Cannot open video")
    quit()

# Get the FPS using OpenCV's property
fps_opencv = cap.get(cv2.CAP_PROP_FPS)
print(f"FPS retrieved using OpenCV: {fps_opencv}")

# Get the total number of frames and video duration to calculate FPS manually
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_duration = total_frames / fps_opencv  # Using the initially retrieved FPS
fps_manual = total_frames / video_duration if video_duration > 0 else 0

print(f"Total frames: {total_frames}")
print(f"Video duration (seconds): {video_duration}")
print(f"Manually calculated FPS: {fps_manual:.2f}")

# Use the more accurate FPS value (if applicable)
fps = fps_manual if fps_manual > 0 else fps_opencv
print(f"Final FPS used: {fps}")

bowling_out_filename = bowling_filename + "_output_test3D.mp4"

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
out_frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out_fps = cap.get(cv2.CAP_PROP_FPS)
print(out_fps)

frame_counter = 0

#Enter following details for Bowling video
#Enter the Ball Release Frame
ball_release_frame = 115
ASSUMED_SHOULDER_WIDTH_M = 0.41  #teen (0.38 to 0.40) + adult (0.40-0.43) but can be adjusted based on player


frame_time = (frame_counter-ball_release_frame)/out_fps
# Create VideoWriter object

out_video = cv2.VideoWriter(bowling_out_filename, fourcc, out_fps, (out_frame_width, out_frame_height))

filename_angles = bowling_filename + "_" + str(ball_release_frame) + "_output_test3D.csv"

# --- ORIGINAL HEADER (kept) ---
Header_row = ["frame_counter", "frame_time",
              "left_arm_angle", "right_arm_angle",
              "left_leg_angle", "right_leg_angle",
              "left_foot_angle", "right_foot_angle",
              "left_wrist_angle", "right_wrist_angle",
              "Speed"]
Header_row += [
    "left_arm_angle_3d", "right_arm_angle_3d",
    "left_leg_angle_3d", "right_leg_angle_3d",
    "left_foot_angle_3d", "right_foot_angle_3d",
    "left_wrist_angle_3d", "right_wrist_angle_3d",
]

Header_row += [
     "right_wrist_speed_3d_mph","left_wrist_speed_3d_mph"
                ]

# --- your existing 3D pose point columns (kept) ---
landmark_names = [lm.name.lower() for lm in mp_pose.PoseLandmark]

with open(filename_angles, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(Header_row)

prev_right_wrist = None
prev_left_wrist = None
max_right_wrist_speed = 0
max_left_wrist_speed = 0
prev_right_wrist_3d = None
prev_left_wrist_3d = None
# --- NEW: max 3D wrist speeds ---
max_right_wrist_speed_3d_mph = 0.0
max_left_wrist_speed_3d_mph = 0.0



## Setup mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5,
                  min_tracking_confidence=0.5,
                  model_complexity=2) as pose:   # NEW (helps 3D stability)

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            print("Error: Unable to capture a frame.")
            break

        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Make detection
        results = pose.process(image)

        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark

            # --- NEW: 3D world landmarks ---
            world_landmarks = None
            if results.pose_world_landmarks:
                world_landmarks = results.pose_world_landmarks.landmark

            def w3d(idx):
            # returns [x,y,z] from world landmarks
                if world_landmarks is None:
                    return None
                lm = world_landmarks[idx]
                return [lm.x, lm.y, lm.z]

            # Get coordinates
            left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            left_elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
            left_wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]

            left_pinky = [landmarks[mp_pose.PoseLandmark.LEFT_PINKY.value].x, landmarks[mp_pose.PoseLandmark.LEFT_PINKY.value].y]
            right_pinky = [landmarks[mp_pose.PoseLandmark.RIGHT_PINKY.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_PINKY.value].y]

            right_shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            right_elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
            right_wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]

            left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
            left_knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
            left_ankle = [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y]

            right_hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
            right_knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
            right_ankle = [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]

            left_foot = [landmarks[mp_pose.PoseLandmark.LEFT_FOOT_INDEX.value].x, landmarks[mp_pose.PoseLandmark.LEFT_FOOT_INDEX.value].y]
            right_foot = [landmarks[mp_pose.PoseLandmark.RIGHT_FOOT_INDEX.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_FOOT_INDEX.value].y]

            # Calculate angle
            left_arm_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
            right_arm_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)

            left_leg_angle = calculate_angle(left_hip, left_knee, left_ankle)
            right_leg_angle = calculate_angle(right_hip, right_knee, right_ankle)

            left_wrist_angle = calculate_angle(left_elbow, left_wrist, left_pinky)
            right_wrist_angle = calculate_angle(right_elbow, right_wrist, right_pinky)

            left_foot_angle = calculate_angle(left_knee, left_ankle, left_foot)
            right_foot_angle = calculate_angle(right_knee, right_ankle, right_foot)

            left_arm_angle = "{:.2f}".format(left_arm_angle)
            right_arm_angle = "{:.2f}".format(right_arm_angle)
            left_leg_angle = "{:.2f}".format(left_leg_angle)
            right_leg_angle = "{:.2f}".format(right_leg_angle)

            left_foot_angle = "{:.2f}".format(left_foot_angle)
            right_foot_angle = "{:.2f}".format(right_foot_angle)

            left_wrist_angle = "{:.2f}".format(left_wrist_angle)
            right_wrist_angle = "{:.2f}".format(right_wrist_angle)

            frame_time = (frame_counter-ball_release_frame)/out_fps
            frame_time = "{:.2f}".format(frame_time)

            # Get coordinates
            left_wrist = np.array([
                landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,
                landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y
            ])
            right_wrist = np.array([
                landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x,
                landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y
            ])
            # -------------------------
            # NEW: 3D angles (world)
            # -------------------------
            LSH = mp_pose.PoseLandmark.LEFT_SHOULDER.value
            LEB = mp_pose.PoseLandmark.LEFT_ELBOW.value
            LWR = mp_pose.PoseLandmark.LEFT_WRIST.value
            LPK = mp_pose.PoseLandmark.LEFT_PINKY.value
            
            RSH = mp_pose.PoseLandmark.RIGHT_SHOULDER.value
            REB = mp_pose.PoseLandmark.RIGHT_ELBOW.value
            RWR = mp_pose.PoseLandmark.RIGHT_WRIST.value
            RPK = mp_pose.PoseLandmark.RIGHT_PINKY.value
            
            LHI = mp_pose.PoseLandmark.LEFT_HIP.value
            LKN = mp_pose.PoseLandmark.LEFT_KNEE.value
            LAN = mp_pose.PoseLandmark.LEFT_ANKLE.value
            LFT = mp_pose.PoseLandmark.LEFT_FOOT_INDEX.value
            
            RHI = mp_pose.PoseLandmark.RIGHT_HIP.value
            RKN = mp_pose.PoseLandmark.RIGHT_KNEE.value
            RAN = mp_pose.PoseLandmark.RIGHT_ANKLE.value
            RFT = mp_pose.PoseLandmark.RIGHT_FOOT_INDEX.value
            
            left_arm_angle_3d = float("nan")
            right_arm_angle_3d = float("nan")
            left_leg_angle_3d = float("nan")
            right_leg_angle_3d = float("nan")
            left_foot_angle_3d = float("nan")
            right_foot_angle_3d = float("nan")
            left_wrist_angle_3d = float("nan")
            right_wrist_angle_3d = float("nan")
            
            if world_landmarks is not None:
                left_arm_angle_3d   = calculate_angle_3d(w3d(LSH), w3d(LEB), w3d(LWR))
                right_arm_angle_3d  = calculate_angle_3d(w3d(RSH), w3d(REB), w3d(RWR))
            
                left_leg_angle_3d   = calculate_angle_3d(w3d(LHI), w3d(LKN), w3d(LAN))
                right_leg_angle_3d  = calculate_angle_3d(w3d(RHI), w3d(RKN), w3d(RAN))
            
                left_wrist_angle_3d = calculate_angle_3d(w3d(LEB), w3d(LWR), w3d(LPK))
                right_wrist_angle_3d= calculate_angle_3d(w3d(REB), w3d(RWR), w3d(RPK))
            
                left_foot_angle_3d  = calculate_angle_3d(w3d(LKN), w3d(LAN), w3d(LFT))
                right_foot_angle_3d = calculate_angle_3d(w3d(RKN), w3d(RAN), w3d(RFT))



                left_arm_angle_3d = "{:.2f}".format(left_arm_angle_3d)
                right_arm_angle_3d = "{:.2f}".format(right_arm_angle_3d)
                left_leg_angle_3d = "{:.2f}".format(left_leg_angle_3d)
                right_leg_angle_3d = "{:.2f}".format(right_leg_angle_3d)
    
                left_foot_angle_3d = "{:.2f}".format(left_foot_angle_3d)
                right_foot_angle_3d = "{:.2f}".format(right_foot_angle_3d)
    
                left_wrist_angle_3d = "{:.2f}".format(left_wrist_angle_3d)
                right_wrist_angle_3d = "{:.2f}".format(right_wrist_angle_3d)
            
            # Calculate 2D wrist speeds
            right_wrist_speed_mph = 0
            left_wrist_speed_mph = 0

            if prev_right_wrist is not None:
                right_wrist_speed = np.linalg.norm(right_wrist - prev_right_wrist) * out_fps
                right_wrist_speed_mph = right_wrist_speed * 2.23694
                if right_wrist_speed_mph < 0.1 or right_wrist_speed_mph > 28:
                    right_wrist_speed_mph = 0

            if prev_left_wrist is not None:
                left_wrist_speed = np.linalg.norm(left_wrist - prev_left_wrist) * out_fps
                left_wrist_speed_mph = left_wrist_speed * 2.23694
                if left_wrist_speed_mph < 0.1 or left_wrist_speed_mph > 28:
                    left_wrist_speed_mph = 0

            prev_right_wrist = right_wrist
            prev_left_wrist = left_wrist
            right_wrist_speed_mph = round(right_wrist_speed_mph, 1)
            left_wrist_speed_mph = round(left_wrist_speed_mph, 1)

            if right_wrist_speed_mph > max_right_wrist_speed:
                max_right_wrist_speed = right_wrist_speed_mph

            if left_wrist_speed_mph > max_left_wrist_speed:
                max_left_wrist_speed = left_wrist_speed_mph

            

            out_row = [frame_counter, frame_time,
                       left_arm_angle, right_arm_angle,
                       left_leg_angle, right_leg_angle,
                       left_foot_angle, right_foot_angle,
                       left_wrist_angle, right_wrist_angle,
                       right_wrist_speed_mph,
                       # NEW 3D angles (write numeric values)
                       left_arm_angle_3d, right_arm_angle_3d,
                       left_leg_angle_3d, right_leg_angle_3d,
                       left_foot_angle_3d, right_foot_angle_3d,
                       left_wrist_angle_3d, right_wrist_angle_3d]



            # =========================
            # NEW: 3D wrist speed (Option A: shoulder marker scaling)
            # =========================
            right_wrist_speed_3d_m_s = 0.0
            left_wrist_speed_3d_m_s = 0.0
            right_wrist_speed_3d_mph = 0.0
            left_wrist_speed_3d_mph = 0.0
            
            if world_landmarks is not None and out_fps > 0:
                # Pull 3D points
                rw3 = w3d(mp_pose.PoseLandmark.RIGHT_WRIST.value)
                lw3 = w3d(mp_pose.PoseLandmark.LEFT_WRIST.value)
                rsh3 = w3d(mp_pose.PoseLandmark.RIGHT_SHOULDER.value)
                lsh3 = w3d(mp_pose.PoseLandmark.LEFT_SHOULDER.value)
            
                if rw3 is not None and lw3 is not None and rsh3 is not None and lsh3 is not None:
                    rw3 = np.array(rw3, dtype=np.float64)
                    lw3 = np.array(lw3, dtype=np.float64)
                    rsh3 = np.array(rsh3, dtype=np.float64)
                    lsh3 = np.array(lsh3, dtype=np.float64)
            
                    # Shoulder distance in 3D (in model "world" units)
                    shoulder_dist = np.linalg.norm(rsh3 - lsh3)
            
                    # Scale factor (Option A): convert world-units -> meters
                    # If shoulder_dist is already close to meters, this will be ~1.
                    scale_to_m = 1.0
                    if shoulder_dist > 1e-6:
                        scale_to_m = ASSUMED_SHOULDER_WIDTH_M / shoulder_dist
            
                    # Right wrist speed
                    if prev_right_wrist_3d is not None:
                        d_rw = np.linalg.norm(rw3 - prev_right_wrist_3d)  # world units per frame
                        right_wrist_speed_3d_m_s = (d_rw * out_fps) * scale_to_m
                        right_wrist_speed_3d_mph = right_wrist_speed_3d_m_s * 2.23694
            
                    # Left wrist speed
                    if prev_left_wrist_3d is not None:
                        d_lw = np.linalg.norm(lw3 - prev_left_wrist_3d)
                        left_wrist_speed_3d_m_s = (d_lw * out_fps) * scale_to_m
                        left_wrist_speed_3d_mph = left_wrist_speed_3d_m_s * 2.23694
            
                    prev_right_wrist_3d = rw3
                    prev_left_wrist_3d = lw3
            
            # (Optional) clamp obvious noise spikes (tune these)
            if right_wrist_speed_3d_mph < 0.1 or right_wrist_speed_3d_mph > 40:
                right_wrist_speed_3d_mph = 0.0
                right_wrist_speed_3d_m_s = 0.0
            if left_wrist_speed_3d_mph < 0.1 or left_wrist_speed_3d_mph > 40:
                left_wrist_speed_3d_mph = 0.0
                left_wrist_speed_3d_m_s = 0.0

            


            # =========================
            # UPDATE MAX 3D WRIST SPEEDS
            # =========================
            
            right_wrist_speed_3d_mph = round(right_wrist_speed_3d_mph, 1)
            max_right_wrist_speed_3d_mph = round(max_right_wrist_speed_3d_mph, 1)
            left_wrist_speed_3d_mph = round(left_wrist_speed_3d_mph, 1)
            max_left_wrist_speed_3d_mph = round(max_left_wrist_speed_3d_mph, 1)
            
            if right_wrist_speed_3d_mph > max_right_wrist_speed_3d_mph:
                max_right_wrist_speed_3d_mph = right_wrist_speed_3d_mph
                      
            if left_wrist_speed_3d_mph > max_left_wrist_speed_3d_mph:
                max_left_wrist_speed_3d_mph = left_wrist_speed_3d_mph

            out_row += [
                        right_wrist_speed_3d_mph,left_wrist_speed_3d_mph
                        ]


            
            with open(filename_angles, mode='a', newline='') as file:
                writer = csv.writer(file)
                writer.writerow(out_row)

            print(frame_time, right_wrist_speed_mph, max_right_wrist_speed, right_wrist_speed_3d_mph, max_right_wrist_speed_3d_mph)
            # Visualize angle
            cv2.putText(image, str("LEFT_ARM " + left_arm_angle),
                        (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

            cv2.putText(image, str("RIGHT_ARM " + right_arm_angle),
                        (10, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

            cv2.putText(image, str("LEFT_WRIST " + left_wrist_angle),
                        (10, 70),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

            cv2.putText(image, str("RIGHT_WRIST " + right_wrist_angle),
                        (10, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

            cv2.putText(image, str("LEFT_LEG " + left_leg_angle),
                        (10, 110),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

            cv2.putText(image, str("RIGHT_LEG " + right_leg_angle),
                        (10, 130),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

            cv2.putText(image, str("FRAME_COUNTER " + str(frame_counter)),
                        (10, 150),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

            cv2.putText(image, str("WRIST_SPEED mph " + str(max_right_wrist_speed)),
                        (10, 170),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

        except:
            pass

        # Render detections (kept)
        if results.pose_landmarks is not None:
            mp_drawing.draw_landmarks(
                image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
            )

        cv2.imshow('Mediapipe Feed', image)
        frame_counter += 1
        out_video.write(image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    df = pd.read_csv(filename_angles)
    print(df.head())
    out_video.release()
    cap.release()
    cv2.destroyAllWindows()

FPS retrieved using OpenCV: 29.97002997002997
Total frames: 188
Video duration (seconds): 6.2729333333333335
Manually calculated FPS: 29.97
Final FPS used: 29.97002997002997
29.97002997002997
-3.84 0 0 0.0 0.0
-3.80 4.8 4.8 18.8 18.8
-3.77 3.9 4.8 21.9 21.9
-3.74 1.1 4.8 5.0 21.9
-3.70 0.3 4.8 0.7 21.9
-3.67 1.8 4.8 27.4 27.4
-3.64 3.0 4.8 6.1 27.4
-3.60 1.5 4.8 5.4 27.4
-3.57 1.2 4.8 2.8 27.4
-3.54 0.4 4.8 1.1 27.4
-3.50 0.3 4.8 0.3 27.4
-3.47 0.5 4.8 1.3 27.4
-3.44 0.4 4.8 0.8 27.4
-3.40 0.6 4.8 0.7 27.4
-3.37 0.6 4.8 2.0 27.4
-3.34 0.7 4.8 1.3 27.4
-3.30 0.3 4.8 0.7 27.4
-3.27 0.3 4.8 0.6 27.4
-3.24 0.5 4.8 1.9 27.4
-3.20 1.1 4.8 2.7 27.4
-3.17 1.1 4.8 2.3 27.4
-3.14 0.3 4.8 0.4 27.4
-3.10 0.4 4.8 0.4 27.4
-3.07 0.6 4.8 0.6 27.4
-3.04 0.7 4.8 1.4 27.4
-3.00 0.7 4.8 0.9 27.4
-2.97 1.9 4.8 3.7 27.4
-2.94 0.4 4.8 0.6 27.4
-2.90 0.5 4.8 0.7 27.4
-2.87 0.8 4.8 1.2 27.4
-2.84 5.9 5.9 10.7 27.4
-2.80 4.2 5.9 17.1 27.4
-2.77 0.5 5.9 9.0 27.4
-2.74 8.8 8.8 17.7 27.4
-2.70 2.1 8.8 6.2 27.4
-2